In [2]:
from git import Repo
from langchain.text_splitter import Language
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain
import os

# Clone the Github Repository

In [4]:
!mkdir -p test_repo

In [6]:
repo_path = 'test_repo'
repo = Repo.clone_from("https://github.com/miteshupadhyay/Doc_RAG_Search",to_path=repo_path)

In [8]:
%pwd

'd:\\GenAI\\RealTimeSourceCodeAnalyzer\\research'

In [10]:
loader = GenericLoader.from_filesystem(repo_path,
                                       glob="**/*.*",
                                       suffixes=[".py"],
                                       parser=LanguageParser(language=Language.PYTHON,parser_threshold=500)
                                       )

In [11]:
documents = loader.load()

In [12]:
documents

[Document(page_content='"""Main application entry point for Agentic RAG system"""\n\nimport sys\nfrom pathlib import Path\n\n# Add src to path\nsys.path.append(str(Path(__file__).parent))\n\nfrom src.config.config import Config\nfrom src.document_ingestion.document_processor import DocumentProcessor\nfrom src.vectorstore.vectorstore import VectorStore\nfrom src.graph_builder.graph_builder import GraphBuilder\n\nclass AgenticRAG:\n    """Main Agentic RAG application"""\n    \n    def __init__(self, urls=None):\n        """\n        Initialize Agentic RAG system\n        \n        Args:\n            urls: List of URLs to process (uses defaults if None)\n        """\n        print("🚀 Initializing Agentic RAG System...")\n        \n        # Use default URLs if none provided\n        self.urls = urls or Config.DEFAULT_URLS\n        \n        # Initialize components\n        self.llm = Config.get_llm()\n        self.doc_processor = DocumentProcessor(\n            chunk_size=Config.CHUNK_SIZ

In [13]:
len(documents)

16

In [14]:
document_spliiter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON,
                                                                 chunk_size=500,
                                                                 chunk_overlap=200)

In [15]:
texts = document_spliiter.split_documents(documents)

In [16]:
len(texts)

67

# Embedding Model

In [17]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [18]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [19]:
embeddings = OpenAIEmbeddings(disallowed_special=())

# Setting up Chroma

In [21]:
vectordb = Chroma.from_documents(texts, embedding=embeddings,persist_directory="./chroma_db")

Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


In [22]:
vectordb.persist()

# Creating an OpenAI Model Wrapper

In [23]:
llm = ChatOpenAI()

In [24]:
memory=ConversationSummaryMemory(llm=llm,memory_key="chat_history",return_messages=True)

In [25]:
qa = ConversationalRetrievalChain.from_llm(llm, retriever=vectordb.as_retriever(search_type="mmr",search_kwargs={"k": 8}),memory=memory)

# Perform Question / Answer

In [26]:
question ="what is the purpose of config.py?"

In [27]:
result = qa(question)

In [28]:
print(result['answer'])

The `config.py` file is used to store configuration parameters for the RAG system. It contains variables that hold values such as API keys, model configurations, document processing parameters, default URLs, and more. These configurations are referenced throughout the system to ensure consistency and easy access to these values.
